### Defining parameters

In [17]:
import pandas as pd

year = 2024
columns_to_keep = [
    "locatie_code",
    "grootheid_code",
    "parameter_code",
    "hoedanigheid_code",
    "eventdatum",
    "event_aquocode",
    "eenheid_code",
]
data_path = rf"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\isc_2023-2025\ISC_{year}.xlsx"
data_delivered_nl_path =  r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\ISC-CIE WGM_Oct 2025_NL.xlsx"
data_delivered_fr_path = r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\ISC-CIE WGM_Tranfert des données RHME 2024_Sept 2025.xlsx"

mapping_location_path = r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\mappings\locations-mapped.xlsx"
mapping_substances_path = r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\mappings\parameter_mapping_final.xlsx"
mapping_hoedanigheid_path = r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\mappings\hoedanigheid_mapped.xlsx"

df_data = pd.read_excel(data_path,sheet_name=str(year))
df_delivered_nl = pd.read_excel(data_delivered_nl_path, sheet_name="Paramètres - Parameters", header=1)
df_delivered_fr = pd.read_excel(data_delivered_fr_path, sheet_name="Paramètres - Parameters", header=1)

df_mapping_location = pd.read_excel(mapping_location_path)
df_mapping_substances = pd.read_excel(mapping_substances_path, sheet_name="total_mapping")
df_mapping_hoedanigheid = pd.read_excel(mapping_hoedanigheid_path)

In [18]:
# Remove from df_mapping_substances any rows where reported column is not nan
df_mapping_substances = df_mapping_substances[df_mapping_substances["reported"].isna()]
df_mapping_substances 

,Unieke identificatie gemeten parameter,CASnummer,AQUO_Codes,AQUO_Omschrijving,ISC_Parameter,Unieke identificatie van de eenheid,grootheid_code,wadar_PARCode,eenheid_code,reported
0,1082,NaN,BaA,benzo(a)antraceen,Benzo(a)antraceen,µg/L,NaN,NaN,NaN,NaN
1,1083,2921-88-2,C2yClprfs,ethylchloorpyrifos,Chloorpyrifos (Chloorpyrifos ethyl),µg/L,CONCTTE,C2yClprfs,ug/l,NaN
2,1101,15972-60-8,alCl,alachloor,Alachloor,µg/L,CONCTTE,alCl,ug/l,NaN
3,1103,309-00-2,aldn,aldrin,Aldrin,µg/L,CONCTTE,aldn,ug/l,NaN
4,1107,1912-24-9,atzne,atrazine,Atrazine,µg/L,CONCTTE,atzne,ug/l,NaN
...,...,...,...,...,...,...,...,...,...,...
108,6561,1763-23-1,PFOS,perfluoroctaansulfonzuur,Perfluor octaanzuur PFOS,µg/L,NaN,NaN,NaN,NaN
109,6616,117-81-7,DEHP,bis(2-ethylhexyl)ftalaat (DEHP),Di(2-ethylhexyl)ftalaat DEHP,µg/L,CONCTTE,DEHP,ug/l,NaN
112,6830,355-46-4,L_PFHxS,perfluorhexaansulfonzuur,PFHxS,µg/L,CONCTTE,L_PFHxS,ug/l,NaN
113,7073,16984-48-8,F,fluoride,Anorganische fluoriden = fluoride opgelost,mg/L,CONCTTE,F,mg/l,NaN


In [19]:
# Get the substances from df_mapping_substances["AQUO_Codes"] that are not in df_data["parameter_code"]
substances_in_mapping_not_in_data = set(df_mapping_substances["AQUO_Codes"]) - set(df_data["parameter_code"])
print("Substances in mapping but not in data:", substances_in_mapping_not_in_data)

Substances in mapping but not in data: {nan, 'TClBen', '4C9yFol', 'pH', 'GELDHD', 'PFOS', 'BZV5', 'HH', 'T', 'sdrin4', 'N2'}


### Defining functions

In [20]:
def mapping_location(df_data, df_mapping):
    # Mapping column names
    original_loc_col = "locatie_code"
    target_loc_col = "Identitication unique de la station"

    # Keep one mapping row per location code
    df_mapping = df_mapping[[original_loc_col, target_loc_col]]

    # Merge station unique ID into df_data on locatie_code
    df_data = df_data.merge(df_mapping, on=original_loc_col, how="left")

    missing_count = df_data[target_loc_col].isna().sum()
    print(f"Missing location data: {missing_count} rows")
    print(f"Shape of df_data after mapping location: {df_data.shape}")
    return df_data

def mapping_substances_by_aquocode(df_data, df_mapping):

    # ds column and equivalent mapping key
    original_parameter_col = "parameter_code"

    # Rename "AQUO_Codes" to "parameter_code"
    df_mapping = df_mapping.rename(columns={"AQUO_Codes": original_parameter_col})

    # Column you want to bring from mapping table
    target_parameter_col = "Unieke identificatie gemeten parameter"  

    df_mapping = df_mapping[[original_parameter_col, target_parameter_col]]
    print(f"Shape of df_mapping: {df_mapping.shape}")

    # # This keeps only rows where parameter exists in the mapping parameter table 
    # df_data = df_data.merge(df_mapping, on=original_parameter_col, how="inner")
    # This keeps all rows and adds the parameter column from the mapping table
    df_data = df_data.merge(df_mapping, on=original_parameter_col, how="left")
    print(f"Shape of df_data after mapping location: {df_data.shape}")
    return df_data

def mapping_substances_by_grootheid(df_data, df_mapping):
    """
    Fill missing 'Unieke identificatie gemeten parameter' values using grootheid_code
    for special grootheid codes only.
    """
    original_grootheid_col = "grootheid_code"
    target_parameter_col = "Unieke identificatie gemeten parameter"

    special_grootheid_codes = ["GELDHD", "HH", "T", "pH"]

    if original_grootheid_col not in df_data.columns:
        raise KeyError(f"Missing column in df_data: {original_grootheid_col}")
    if original_grootheid_col not in df_mapping.columns:
        raise KeyError(f"Missing column in df_mapping: {original_grootheid_col}")
    if target_parameter_col not in df_mapping.columns:
        raise KeyError(f"Missing column in df_mapping: {target_parameter_col}")

    # Keep only the special grootheid codes in the mapping table
    df_mapping = df_mapping[[original_grootheid_col, target_parameter_col]].copy()
    df_mapping = df_mapping[df_mapping[original_grootheid_col].isin(special_grootheid_codes)]

    # Avoid duplicate merge rows
    df_mapping = df_mapping.drop_duplicates(subset=[original_grootheid_col])

    print(f"Shape of grootheid mapping: {df_mapping.shape}")

    # Merge without overwriting existing parameter ids
    df_data = df_data.merge(
        df_mapping,
        on=original_grootheid_col,
        how="left",
        suffixes=("", "_grootheid"),
    )

    # Fill only rows that still do not have a unique parameter id
    df_data[target_parameter_col] = df_data[target_parameter_col].fillna(
        df_data[f"{target_parameter_col}_grootheid"]
    )

    # Drop helper column
    df_data = df_data.drop(columns=[f"{target_parameter_col}_grootheid"])

    print(f"Shape of df_data after grootheid mapping: {df_data.shape}")
    return df_data

def mapping_hoedanigheid(df_data, df_mapping):
    # Rename hoedanigheid_code_wadar to hoedanigheid_code in the mapping table
    df_mapping = df_mapping.rename(columns={"hoedanigheid_code_wadar": "hoedanigheid_code"})

    # Mapping column names
    original_hoedanigheid_col = "hoedanigheid_code"
    target_hoedanigheid_col = "ISC_fraction"

    # Keep one mapping row per hoedanigheid code
    df_mapping = df_mapping[[original_hoedanigheid_col, target_hoedanigheid_col]]

    # Merge hoedanigheid unique ID into df_data on hoedanigheid_code
    df_data = df_data.merge(df_mapping, on=original_hoedanigheid_col, how="left")
    print(f"Shape of df_data after mapping hoedanigheid: {df_data.shape}")
    return df_data

def drop_rows_without_parameter(df_data):
    target_parameter_col = "Unieke identificatie gemeten parameter"
    df_data = df_data[df_data[target_parameter_col].notna()]
    print(f"Shape of df_data after dropping rows without parameter: {df_data.shape}")
    return df_data

def filter_data(df_data):
    #Removing df_data where waardebewerkings_methode_code is BER then print shape of filtered DataFrame
    filtered_df = df_data[df_data["waardebewerkings_methode_code"] != "BER"]
    print("Shape of filtered DataFrame (excluding BER):", filtered_df.shape)

    # Remove rows where bemonsteringshoogte_code is different -100 then print shape of filtered DataFrame
    filtered_df = filtered_df[filtered_df["bemonsteringshoogte_code"] == -100]
    print("Shape of filtered DataFrame (bemonsteringshoogte_code == -100):", filtered_df.shape)

    # Remove rwos where event_aquocode is differente to 0, 3, 90, or 99 and print shape of filtered DataFrame
    valid_aquocodes = [0, 3, 90, 99]
    filtered_df = filtered_df[filtered_df["event_aquocode"].isin(valid_aquocodes)]
    print("Shape of filtered DataFrame (valid event_aquocode):", filtered_df.shape)

    # If target parameter column is 1369.0, 1379.0, 1382.0, 1386.0, 1387.0, 1388.0, 1392.0, 1361, 1368, 1383, 1385 then remove those where target hoedanigheid column is EB
    target_parameter_col = "Unieke identificatie gemeten parameter"
    target_hoedanigheid_col = "ISC_fraction"
    specific_parameters = [1369.0, 1379.0, 1382.0, 1386.0, 1387.0, 1388.0, 1392.0, 1361, 1368, 1383, 1385]
    filtered_df = filtered_df[~((filtered_df[target_parameter_col].isin(specific_parameters)) & (filtered_df[target_hoedanigheid_col] == "EB"))]
    print("Shape of filtered DataFrame (excluding specific parameters with EB):", filtered_df.shape)

    # If ISC_fraction is "not reported" remove those rows and print shape of filtered DataFrame
    filtered_df = filtered_df[filtered_df[target_hoedanigheid_col] != "not reported"]
    print("Shape of filtered DataFrame (excluding not reported ISC_fraction):", filtered_df.shape)

    return filtered_df

def select_lowest_aquocode_with_notes(df_data):
    """
    For rows with identical location, date, hoedanigheid, parameter, and eenheid,
    keep the one with the lowest valid aquocode and note alternative aquocodes.
    
    Parameters:
    -----------
    df_data : DataFrame
        Input dataframe (typically filtered_df)
    
    Returns:
    --------
    tuple : (kept_df, removed_df)
        kept_df: Dataframe with duplicate aquocodes resolved, plus new columns for alternatives
        removed_df: Dataframe with all rows that were removed/not selected
    """
    # Hardcoded column names
    target_loc_col = "Identitication unique de la station"
    target_hoedanigheid_col = "ISC_fraction"
    target_parameter_col = "Unieke identificatie gemeten parameter"
    
    valid_aquocodes = [0, 3, 90, 99]
    
    # Group by the common columns
    group_cols = [
        target_loc_col, 
        "eventdatum", 
        target_hoedanigheid_col, 
        target_parameter_col, 
        "eenheid_code"
    ]
    
    kept_rows = []
    removed_rows = []
    
    for group_key, group_df in df_data.groupby(group_cols, dropna=False):
        # Get all aquocodes in this group
        aquocodes_in_group = group_df["event_aquocode"].tolist()
        
        # Find the lowest valid aquocode
        valid_in_group = [ac for ac in aquocodes_in_group if ac in valid_aquocodes]
        lowest_aquocode = min(valid_in_group) if valid_in_group else None
        
        # Select row with lowest aquocode
        row_to_keep = group_df[group_df["event_aquocode"] == lowest_aquocode].iloc[0].copy()
        
        # Find higher aquocodes that were present
        higher_aquocodes = [ac for ac in valid_aquocodes if ac > lowest_aquocode]
        higher_present = {ac: aquocodes_in_group.count(ac) for ac in higher_aquocodes if ac in aquocodes_in_group}
        
        # Add columns for alternative aquocodes
        row_to_keep["has_higher_aquocodes"] = "yes" if higher_present else "no"
        row_to_keep["higher_aquocodes_info"] = str(higher_present) if higher_present else ""
        
        kept_rows.append(row_to_keep)
        
        # Add removed rows to removed_rows list
        rows_removed = group_df[group_df["event_aquocode"] != lowest_aquocode]
        if len(rows_removed) > 0:
            removed_rows.extend(rows_removed.to_dict('records'))
    
    kept_df = pd.DataFrame(kept_rows).reset_index(drop=True)
    removed_df = pd.DataFrame(removed_rows).reset_index(drop=True)
    
    print(f"Shape of result after selecting lowest aquocode: {kept_df.shape}")
    print(f"Shape of removed rows: {removed_df.shape}")
    
    return kept_df, removed_df

def get_count_unique_parameter_values(filtered_df):
    target_parameter_col = "Unieke identificatie gemeten parameter"
    # Get count of unique values in the target column after merge
    unique_values_after_merge = filtered_df[target_parameter_col].value_counts(dropna=False)
    print(unique_values_after_merge)
    print(f"Shape of unique_values_after_merge: {unique_values_after_merge.shape}")
    return unique_values_after_merge

def check_duplicates(filtered_df):
    target_loc_col = "Identitication unique de la station"
    target_parameter_col = "Unieke identificatie gemeten parameter"
    target_hoedanigheid_col = "ISC_fraction"
    # Keep only the columns of interest for duplicate checking
    columns_to_keep = [target_loc_col ,"grootheid_code",target_parameter_col , target_hoedanigheid_col, "eventdatum", "event_aquocode", "eenheid_code"]
    filtered_df_duplicates = filtered_df[columns_to_keep].copy()
    filtered_df_duplicates 

    # Check if there are duplicated rows in the filtered DataFrame
    has_identical_rows = filtered_df_duplicates.duplicated().any()
    print("Identical rows found:" if has_identical_rows else "No identical rows found.")

def split_by_station(df_data):
    target_loc_col = "Identitication unique de la station"
    dfs_by_station = {
        station_id: group_df.reset_index(drop=True)
        for station_id, group_df in df_data.groupby(target_loc_col, dropna=False)
    }
    print(f"Created {len(dfs_by_station)} station DataFrames")
    return dfs_by_station

def get_repeated_cases_by_station(dfs_by_station):
    target_loc_col = "Identitication unique de la station"
    target_parameter_col = "Unieke identificatie gemeten parameter"
    columns_to_keep = [
        target_loc_col,
        "grootheid_code",
        target_parameter_col,
        "ISC_fraction",
        "eventdatum",
        "event_aquocode",
        "eenheid_code",
    ]
    case_cols = [c for c in columns_to_keep if c != "eventdatum"]

    repeated_cases_by_station = {}

    for station_id, station_df in dfs_by_station.items():
        station_df_duplicates = station_df[columns_to_keep].copy()

        repeated_cases = (
            station_df_duplicates
            .groupby(case_cols, dropna=False)
            .agg(
                n_measurements=("eventdatum", "size"),
                n_unique_dates=("eventdatum", "nunique"),
                first_date=("eventdatum", "min"),
                last_date=("eventdatum", "max"),
            )
            .reset_index()
        )

        repeated_cases = repeated_cases[repeated_cases["n_measurements"] >= 1]
        repeated_cases = repeated_cases.sort_values("n_measurements", ascending=False)

        repeated_cases_by_station[station_id] = repeated_cases.reset_index(drop=True)

    print(f"Created {len(repeated_cases_by_station)} repeated-case tables")
    return repeated_cases_by_station

def harmonized_df(df_data):
    target_loc_col = "Identitication unique de la station"
    target_hoedanigheid_col = "ISC_fraction"
    target_parameter_col = "Unieke identificatie gemeten parameter"

    columns_to_keep = [
        target_loc_col,
        "eventdatum",
        target_hoedanigheid_col,
        target_parameter_col,
        "event_waarde_limietsymbool",
        "event_waarde",
        "eenheid_code",
        "event_aquocode",
    ]

    missing_cols = [c for c in columns_to_keep if c not in df_data.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    result_df = df_data[columns_to_keep].copy()

    # Parse date once for reliable sorting and formatting
    result_df["eventdatum"] = pd.to_datetime(result_df["eventdatum"], errors="coerce")

    # Sort rows: location -> date -> substance(parameter)
    result_df = result_df.sort_values(
        by=[target_loc_col, "eventdatum", target_parameter_col],
        ascending=[True, True, True],
        na_position="last",
        kind="mergesort"
    ).reset_index(drop=True)

    # Format date as dd/mm/jjjj
    result_df["eventdatum"] = result_df["eventdatum"].dt.strftime("%d/%m/%Y")

    # Harmonize output names
    rename_map = {
        target_loc_col: "Identitication unique de la station",
        "eventdatum": "Date de prélévement",
        target_hoedanigheid_col: "Fraction analysée",
        target_parameter_col: "Identification unique du paramètre mesuré",
        "event_waarde_limietsymbool": "Gestion de la Limite de Quantification",
        "event_waarde": "Résultat",
        "eenheid_code": "Identification unique de l'unité",
    }
    result_df = result_df.rename(columns=rename_map)

    print(f"Shape of harmonized dataframe: {result_df.shape}")
    return result_df

def get_repeated_cases_by_station(dfs_by_station):
    target_loc_col = "Identitication unique de la station"
    target_parameter_col = "Unieke identificatie gemeten parameter"
    target_hoedanigheid_col = "ISC_fraction"
    columns_to_keep = [
        target_loc_col,
        target_parameter_col,
        target_hoedanigheid_col,
        "eventdatum",
        "event_aquocode",
        "eenheid_code",
    ]
    case_cols = [c for c in columns_to_keep if c != "eventdatum"]

    repeated_cases_by_station = {}

    for station_id, station_df in dfs_by_station.items():
        station_df_duplicates = station_df[columns_to_keep].copy()

        # normalize to calendar day (ignore time part)
        station_df_duplicates["event_day"] = pd.to_datetime(
            station_df_duplicates["eventdatum"], errors="coerce"
        ).dt.normalize()

        def same_day_duplicate_info(group):
            # count how many measurements per day within this case
            day_counts = group["event_day"].value_counts(dropna=True)

            # days with more than 1 measurement
            has_same_day = (day_counts > 1).any()

            # unique-parameter ids that appear on a duplicated day
            # (for this grouping, it will usually be the same id, but kept generic)
            if has_same_day:
                ids = sorted(group[target_parameter_col].dropna().astype(str).unique())
                ids_text = ", ".join(ids)
            else:
                ids_text = ""

            return pd.Series(
                {
                    "n_measurements": group["eventdatum"].size,
                    "n_unique_dates": group["eventdatum"].nunique(),
                    "first_date": group["eventdatum"].min(),
                    "last_date": group["eventdatum"].max(),
                    "same_day_measurement": "yes" if has_same_day else "no",
                    "same_day_parameter_ids": ids_text,
                }
            )
        
        value_cols = ["eventdatum", "event_day", target_parameter_col]

        repeated_cases = (
            station_df_duplicates
            .groupby(case_cols, dropna=False)[value_cols]
            .apply(same_day_duplicate_info)
            .reset_index()
        )

        repeated_cases = repeated_cases[repeated_cases["n_measurements"] >= 1]
        repeated_cases = repeated_cases.sort_values("n_measurements", ascending=False)

        repeated_cases_by_station[station_id] = repeated_cases.reset_index(drop=True)

    print(f"Created {len(repeated_cases_by_station)} repeated-case tables")
    return repeated_cases_by_station

def print_parameter_counts_by_event_aquocode(
    repeated_cases_by_station,
    valid_aquocodes=None,
    show_only_duplicates=True,
    top_n=None,
):
    if valid_aquocodes is None:
        valid_aquocodes = [0, 3, 90, 99]

    target_parameter_col = "Unieke identificatie gemeten parameter"
    target_hoedanigheid_col = "ISC_fraction"
    same_day_col = "same_day_measurement"  # expects values like "yes"/"no"

    grand_total_cases = 0
    grand_total_same_day_yes = 0

    def print_parameter_summary(df, indent="      "):
        counts = df[target_parameter_col].value_counts(ascending=False)

        if show_only_duplicates:
            counts = counts[counts > 1]

        if top_n is not None:
            counts = counts.head(top_n)

        if len(counts) == 0:
            print(f"{indent}No duplicate parameters")
            return

        for parameter_id, count in counts.items():
            print(f"{indent}parameter {parameter_id}: {count} cases")

    for station_id, station_df in repeated_cases_by_station.items():
        print("\n" + "=" * 70)
        print(f"STATION: {station_id}")
        print("=" * 70)

        station_total_cases = 0
        station_total_same_day_yes = 0

        for event_aquocode in valid_aquocodes:
            df_aquocode = station_df[station_df["event_aquocode"] == event_aquocode]
            n_cases = len(df_aquocode)
            n_unique_parameters = df_aquocode[target_parameter_col].nunique()
            station_total_cases += n_cases

            # Count rows/cases flagged as yes for same-day measurements
            n_same_day_yes = (
                df_aquocode[same_day_col]
                .astype(str)
                .str.lower()
                .eq("yes")
                .sum()
            )
            station_total_same_day_yes += n_same_day_yes

            print(f"\n  AQUOCODE {event_aquocode}")
            print(f"  Cases: {n_cases} | Unique parameters: {n_unique_parameters}")
            print(f"  Cases with same-day measurements (yes): {n_same_day_yes}")

            if n_cases == 0:
                print("  No rows")
                continue

            # Aquocode level
            print("  Summary (all hoedanigheid_code):")
            print_parameter_summary(df_aquocode, indent="    ")

            # Hoedanigheid level
            hoedanigheid_groups = (
                df_aquocode
                .groupby(target_hoedanigheid_col, dropna=False)
                .size()
                .sort_values(ascending=False)
            )

            print("  By hoedanigheid_code:")
            for hoedanigheid_code in hoedanigheid_groups.index:
                df_h = df_aquocode[df_aquocode[target_hoedanigheid_col] == hoedanigheid_code]
                n_cases_h = len(df_h)
                n_unique_h = df_h[target_parameter_col].nunique()
                n_same_day_yes_h = (
                    df_h[same_day_col]
                    .astype(str)
                    .str.lower()
                    .eq("yes")
                    .sum()
                )

                print(
                    f"    - {hoedanigheid_code}: "
                    f"{n_cases_h} cases, {n_unique_h} unique parameters, "
                    f"{n_same_day_yes_h} with same-day measurements (yes)"
                )
                print_parameter_summary(df_h, indent="      ")

        print(f"\n  STATION TOTAL: {station_total_cases} cases")
        print(f"  STATION TOTAL with same-day measurements (yes): {station_total_same_day_yes} cases")

        grand_total_cases += station_total_cases
        grand_total_same_day_yes += station_total_same_day_yes

    print("\n" + "=" * 70)
    print(f"GRAND TOTAL: {grand_total_cases} cases")
    print(f"GRAND TOTAL with same-day measurements (yes): {grand_total_same_day_yes} cases")
    print("=" * 70)


### Running tasks

In [21]:
df_data = mapping_location(df_data, df_mapping_location)
df_data = mapping_substances_by_aquocode(df_data, df_mapping_substances)
df_data = mapping_substances_by_grootheid(df_data, df_mapping_substances)
df_data = mapping_hoedanigheid(df_data, df_mapping_hoedanigheid)
df_data = drop_rows_without_parameter(df_data)
filtered_df = filter_data(df_data)
filtered_df, removed_df = select_lowest_aquocode_with_notes(filtered_df)

# Select filtered_df where higher_aquocodes_info is not empty (i.e. there were higher aquocodes present for the same case)  
filtered_df_with_higher_aquocodes = filtered_df[filtered_df["higher_aquocodes_info"] != ""]

unique_values_after_merge = get_count_unique_parameter_values(filtered_df)
check_duplicates(filtered_df)
harmonized_df = harmonized_df(filtered_df)

# Explore results 
dfs_by_station = split_by_station(filtered_df)
repeated_cases_by_station = get_repeated_cases_by_station(dfs_by_station)
print_parameter_counts_by_event_aquocode(repeated_cases_by_station)



Missing location data: 0 rows
Shape of df_data after mapping location: (33403, 45)
Shape of df_mapping: (101, 2)
Shape of df_data after mapping location: (33641, 46)
Shape of grootheid mapping: (4, 2)
Shape of df_data after grootheid mapping: (33641, 46)
Shape of df_data after mapping hoedanigheid: (33641, 47)
Shape of df_data after dropping rows without parameter: (9961, 47)
Shape of filtered DataFrame (excluding BER): (9961, 47)
Shape of filtered DataFrame (bemonsteringshoogte_code == -100): (7386, 47)
Shape of filtered DataFrame (valid event_aquocode): (7386, 47)
Shape of filtered DataFrame (excluding specific parameters with EB): (6580, 47)
Shape of filtered DataFrame (excluding not reported ISC_fraction): (6580, 47)


C:\Users\fuentesm\AppData\Local\Temp\ipykernel_36124\3138892708.py:73: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_data[target_parameter_col] = df_data[target_parameter_col].fillna(


Shape of result after selecting lowest aquocode: (6388, 49)
Shape of removed rows: (2, 47)
Unieke identificatie gemeten parameter
1311.0    190
1305.0     95
1302.0     95
1335.0     95
1340.0     95
         ... 
5978.0     52
6830.0     52
5979.0     52
7073.0     32
1289.0      3
Name: count, Length: 93, dtype: int64
Shape of unique_values_after_merge: (93,)
No identical rows found.
Shape of harmonized dataframe: (6388, 8)
Created 5 station DataFrames
Created 5 repeated-case tables

STATION: NL89_SCHAARVODDL

  AQUOCODE 0
  Cases: 85 | Unique parameters: 84
  Cases with same-day measurements (yes): 0
  Summary (all hoedanigheid_code):
    parameter 1311.0: 2 cases
  By hoedanigheid_code:
    - EB: 67 cases, 66 unique parameters, 0 with same-day measurements (yes)
      parameter 1311.0: 2 cases
    - EF: 17 cases, 17 unique parameters, 0 with same-day measurements (yes)
      No duplicate parameters
    - nan: 0 cases, 0 unique parameters, 0 with same-day measurements (yes)
      No

In [22]:
# Save the harmonized dataframe to an Excel file
output_path = rf"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\isc_2023-2025\ISC_{year}_harmonized.csv"
harmonized_df.to_csv(output_path, index=False, sep=";", encoding="utf-8-sig")



In [23]:
# Save filtered df to an Excel file
filtered_output_path = rf"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\isc_2023-2025\ISC_{year}_filtered.csv"
filtered_df.to_csv(filtered_output_path, index=False, sep=";", encoding="utf-8-sig")

In [24]:
# Pick only the columns you want
cols = [
    "grootheid_code",
    "parameter_code",
    "hoedanigheid_code",
    "Unieke identificatie gemeten parameter",
    "eenheid_code"
]

# Keep those columns, then remove duplicate rows
filtered_df_dedup = (
    filtered_df[cols]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(filtered_df_dedup.shape)
filtered_df_dedup.head()

(94, 5)


,grootheid_code,parameter_code,hoedanigheid_code,Unieke identificatie gemeten parameter,eenheid_code
0,T,NVT,NVT,1301.0,oC
1,pH,NVT,NVT,1302.0,DIMSLS
2,CONCTTE,OS,NVT,1305.0,mg/l
3,VERZDGGD,O2,NVT,1311.0,%
4,CONCTTE,O2,NVT,1311.0,mg/l


In [26]:
filtered_df_dedup
# Save the deduplicated filtered df to an Excel file
dedup_output_path = rf"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\isc_2023-2025\ISC_{year}_dedup.csv"

filtered_df_dedup.to_csv(dedup_output_path, index=False, sep=";", encoding="utf-8-sig")